<a href="https://colab.research.google.com/github/sck2189985/econ8310-assignment3-baseball/blob/main/assignment_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3
## Econ 8310 - Business Forecasting

For homework assignment 3, you will work with [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist), a more fancier data set.

- You must create a custom data loader as described in the first week of neural network lectures [2 points]
    - You will NOT receive credit for this if you use the pytorch prebuilt loader for Fashion MNIST!
- You must create a working and trained neural network using only pytorch [2 points]
- You must store your weights and create an import script so that I can evaluate your model without training it [2 points]

Highest accuracy score gets some extra credit!

Submit your forked repository URL on Canvas! :) I'll be manually grading this assignment.

Some checks you can make on your own:
- Did you manually process the data or use a prebuilt loader (see above)?
- Does your script train a neural network on the assigned data?
- Did your script save your model?
- Do you have separate code to import your model for use after training?

In [11]:
import os
import glob
import random
import cv2
import torch
import torch.nn as nn
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader

In [12]:
# using another student's suggestion to extract frames first
def extract_frames():
    if not os.path.exists("Frames"):
        os.makedirs("Frames")
    video_files = glob.glob("Videos/*.mov")

    for video in video_files:
        base_name = os.path.basename(video)
        name_without_extension = os.path.splitext(base_name)[0]
        cap = cv2.VideoCapture(video)
        frame_number = 0
        while True:
            success, frame = cap.read()
            if success == False:
                break

            frame_filename = "Frames/" + name_without_extension + "_frame" + str(frame_number) + ".jpg"
            cv2.imwrite(frame_filename, frame)
            frame_number = frame_number + 1
        cap.release()

        print("Done extracting:", video)

# run function, then comment this out when done
#extract_frames()

In [ ]:
class BaseballDataset(Dataset):
    def __init__(self):
        self.samples = []
        xml_files = glob.glob("Annotations/*.xml")
        for xml_file in xml_files:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            source_text = root.find(".//source").text.strip()
            video_name = os.path.splitext(source_text)[0]
            tracks = root.findall("track")

            for track in tracks:
                boxes = track.findall("box")

                for box in boxes:
                    # skip frames where baseball isnt visible
                    if box.get("outside") == "1":
                        continue
                    frame_number = box.get("frame")
                    frame_path = "Frames/" + video_name + "_frame" + frame_number + ".jpg"
                    xtl = float(box.get("xtl"))
                    ytl = float(box.get("ytl"))
                    xbr = float(box.get("xbr"))
                    ybr = float(box.get("ybr"))
                    bbox = [xtl, ytl, xbr, ybr]
                    sample = {
                        "frame_path": frame_path,
                        "bbox": bbox
                    }

                    self.samples.append(sample)

    def __len__(self):
        total_length = len(self.samples) * 2
        return total_length

    def _crop(self, image, x1, y1, x2, y2):
        height, width = image.shape[:2]
        x1 = max(0, int(x1))
        y1 = max(0, int(y1))
        x2 = min(width, int(x2))
        y2 = min(height, int(y2))
        crop = image[y1:y2, x1:x2]
        if crop.size == 0:
            crop = image[0:64, 0:64]
        # resize and convert color
        crop = cv2.resize(crop, (64, 64))
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        # convert to tensor
        crop_tensor = torch.tensor(crop, dtype=torch.float32)
        crop_tensor = crop_tensor.permute(2, 0, 1) / 255.0

        return crop_tensor

    def _negative_crop(self, image, bbox):
        height, width = image.shape[:2]

        xtl, ytl, xbr, ybr = bbox
        boxwidth = int(xbr - xtl)
        boxheight = int(ybr - ytl)
        boxwidth = max(12, boxwidth)
        boxheight = max(12, boxheight)

        for attempt in range(20):
            randx1 = random.randint(0, max(0, width - boxwidth - 1))
            randy1 = random.randint(0, max(0, height - boxheight - 1))

            randx2 = randx1 + boxwidth
            randy2 = randy1 + boxheight


            overlap_width = max(0, min(randx2, int(xbr)) - max(randx1, int(xtl)))
            overlap_height = max(0, min(randy2, int(ybr)) - max(randy1, int(ytl)))
            if overlap_width * overlap_height == 0:
                return self._crop(image, randx1, randy1, randx2, randy2)
        return self._crop(image, 0, 0, boxwidth, boxheight)

    def __getitem__(self, idx):
        sample_index = idx // 2
        sample = self.samples[sample_index]
        image = cv2.imread(sample["frame_path"])
        xtl, ytl, xbr, ybr = sample["bbox"]

        #even index, positive sample
        if idx % 2 == 0:
            patch = self._crop(image, xtl, ytl, xbr, ybr)
            label = torch.tensor(1, dtype=torch.long)
        #odd index, negative sample
        else:
            patch = self._negative_crop(image, sample["bbox"])
            label = torch.tensor(0, dtype=torch.long)

        return patch, label
# i struggled with this part, i used he

In [ ]:
# creating the model
class BaseballNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 64 * 64, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# building the custom data loader
class BaseballDataLoader:
    def __init__(self, dataset, batch_size=16, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = list(range(len(dataset)))

    def __iter__(self):
        if self.shuffle:
            random.shuffle(self.indices)
        self.current = 0
        return self

    def __next__(self):
        if self.current >= len(self.indices):
            raise StopIteration
        batch_indices = self.indices[self.current:self.current + self.batch_size]
        batch_images = []
        batch_labels = []

        for idx in batch_indices:
            image, label = self.dataset[idx]
            batch_images.append(image)
            batch_labels.append(label)

        self.current += self.batch_size

        # stack into tensors
        batch_images = torch.stack(batch_images)
        batch_labels = torch.tensor(batch_labels)

        return batch_images, batch_labels

In [ ]:
# training & testing!
def train_model(data_loader, model, loss_function, optimizer, device):
    model.train()
    total_loss = 0
    batch_count = 0
    total_batches = len(data_loader.indices) // data_loader.batch_size

    for batch_images, batch_labels in data_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        predictions = model(batch_images)
        loss = loss_function(predictions, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_count % 5 == 0:
            print(f"Batch {batch_count}/{total_batches} | Loss: {loss.item():.4f}")

        batch_count += 1

    avg_loss = total_loss / batch_count
    print(f"Average Training Loss: {avg_loss:.4f}")

def evaluate_model(data_loader, model, loss_function, device):
    model.eval()
    total_loss = 0
    total_correct = 0

    with torch.no_grad():
        for batch_images, batch_labels in data_loader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            predictions = model(batch_images)
            total_loss += loss_function(predictions, batch_labels).item()
            total_correct += (predictions.argmax(dim=1) == batch_labels).sum().item()

    accuracy = total_correct / len(data_loader.indices)
    avg_loss = total_loss / (len(data_loader.indices) // data_loader.batch_size + 1)
    print(f"Accuracy: {accuracy*100:.1f}%  Loss: {avg_loss:.4f}")

random.seed(42)
torch.manual_seed(42)


dataset = BaseballDataset()

print("Total samples:", len(dataset))

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = BaseballDataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = BaseballDataLoader(test_dataset, batch_size=16, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = BaseballNet().to(device)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 6
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    print("---------------------------")
    train_model(train_loader, model, loss_function, optimizer, device)
    evaluate_model(test_loader, model, loss_function, device)

torch.save(model.state_dict(), "final_model.pt")

Total samples: 496
Using device: cpu
Epoch 1/10
----------------------------
Batch 0/24 | Loss: 0.7003
Batch 5/24 | Loss: 0.2370
Batch 10/24 | Loss: 0.1472
Batch 15/24 | Loss: 0.0210
Batch 20/24 | Loss: 0.0379
Average Training Loss: 0.1959
Accuracy: 100.0%  Loss: 0.0090
Epoch 2/10
----------------------------
Batch 0/24 | Loss: 0.0164
Batch 5/24 | Loss: 0.0011
Batch 10/24 | Loss: 0.0690
Batch 15/24 | Loss: 0.0003
Batch 20/24 | Loss: 0.0067
Average Training Loss: 0.0572
Accuracy: 100.0%  Loss: 0.0071
Epoch 3/10
----------------------------
Batch 0/24 | Loss: 0.0500
Batch 5/24 | Loss: 0.0361
Batch 10/24 | Loss: 0.0102
Batch 15/24 | Loss: 0.0027
Batch 20/24 | Loss: 0.0237
Average Training Loss: 0.1118
Accuracy: 99.0%  Loss: 0.0229
Epoch 4/10
----------------------------
Batch 0/24 | Loss: 0.0093
Batch 5/24 | Loss: 0.1071
Batch 10/24 | Loss: 0.0099
Batch 15/24 | Loss: 0.0017
Batch 20/24 | Loss: 0.0003
Average Training Loss: 0.0731
Accuracy: 98.0%  Loss: 0.0483
Epoch 5/10
------------------

KeyboardInterrupt: 